# Week 16, Day 4: Permutations and subsets

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w016d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
toppings = ["cheese", "ham", "olives"]
count = 0
for a in (False, True):
    for b in (False, True):
        for c in (False, True):
            count += 1
print(count)

In [ ]:
def subsets(items):
    if not items:
        return [[]]
    rest = subsets(items[1:])
    return rest + [[items[0]] + s for s in rest]

for s in subsets(["a", "b", "c"]):
    print(s)

*Interactive figure on the website: Each call splits on the first item. The tree has one leaf per subset..*

In [ ]:
def subsets(items):
    found = []
    def go(i, chosen):
        if i == len(items):
            found.append(chosen)
            return
        go(i + 1, chosen)                  # leave items[i] out
        go(i + 1, chosen + [items[i]])     # put items[i] in
    go(0, [])
    return found

print(subsets([1, 2, 3]))
print(len(subsets(list(range(10)))))

In [ ]:
def permutations(items):
    if len(items) <= 1:
        return [items[:]]
    result = []
    for i in range(len(items)):
        first = items[i]
        others = items[:i] + items[i + 1:]
        for p in permutations(others):
            result.append([first] + p)
    return result

for p in permutations(["Ada", "Ben", "Cy"]):
    print(p)

In [ ]:
from math import factorial

for n in (3, 5, 10, 15, 20):
    print(n, 2 ** n, factorial(n))

In [ ]:
def orderings(items):
    found = []
    def go(chosen, used):
        if len(chosen) == len(items):
            found.append(chosen)
            return
        for i, item in enumerate(items):
            if i not in used:
                go(chosen + [item], used | {i})
    go([], set())
    return found

print(orderings("xyz"))

In [ ]:
def choose(items, k):
    if k == 0:
        return [[]]
    if len(items) < k:
        return []
    first, rest = items[0], items[1:]
    with_first = [[first] + c for c in choose(rest, k - 1)]
    without_first = choose(rest, k)
    return with_first + without_first

print(choose(["Ada", "Ben", "Cy", "Dee"], 2))

> **Key idea.** To generate every possibility: make one choice, recurse on what's left, and collect the results. The base case is "no more choices", which gives one answer (the choices made), not zero.

In [ ]:
def exact_sums(coins, target):
    found = []
    def go(i, chosen, total):
        if total == target:
            found.append(chosen)
            return
        if total > target or i == len(coins):
            return                 # dead end: give up on this path
        go(i + 1, chosen + [coins[i]], total + coins[i])
        go(i + 1, chosen, total)
    go(0, [], 0)
    return found

print(exact_sums([5, 10, 2, 3, 8], 13))

In [ ]:
def queens(n):
    solutions = []
    def go(row, cols):
        if row == n:
            solutions.append(cols)
            return
        for c in range(n):
            safe = all(c != c2 and abs(c - c2) != row - r2 for r2, c2 in enumerate(cols))
            if safe:
                go(row + 1, cols + [c])
    go(0, [])
    return solutions

answers = queens(6)
print(len(answers), "solutions; the first:", answers[0])
for c in answers[0]:
    print(" ".join("Q" if i == c else "." for i in range(6)))

> **Common mistake.** When you build `chosen` by appending to one shared list, every answer you save is the **same** list, which keeps changing. Either pass a new list down (`chosen + [x]`, as here) or save a copy (`found.append(chosen[:])`). Bug 1 below is this mistake.

In [ ]:
from itertools import permutations, combinations, product

print(list(permutations("abc")))
print(list(permutations("abcd", 2)))      # ordered pairs
print(list(combinations([1, 2, 3, 4], 2)))
print(list(product("HT", repeat=3)))       # every outcome of 3 coin tosses
print(list(product([1, 2], "xy")))         # every pair, one from each

In [ ]:
from itertools import combinations

items = ["a", "b", "c"]
everything = [c for k in range(len(items) + 1) for c in combinations(items, k)]
print(everything)

### Exercise: Every bit string

Write `bit_strings(n)` that returns every string of n characters made of `"0"` and `"1"`, in increasing order (`"00"`, `"01"`, `"10"`, `"11"` for n = 2). Use recursion, not `itertools`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert bit_strings(0) == [\"\"]\n    assert bit_strings(1) == [\"0\", \"1\"]\n    assert bit_strings(3) == [\"000\", \"001\", \"010\", \"011\", \"100\", \"101\", \"110\", \"111\"]\n    assert len(bit_strings(10)) == 1024\n    assert \"itertools\" not in source")

### Exercise: Subsets with a size

Write `subsets_of_size(items, k)` returning every subset of exactly k items, keeping the original order within each, in the order that the include-first recursion produces (so `subsets_of_size([1, 2, 3], 2)` is `[[1, 2], [1, 3], [2, 3]]`). Don't use `itertools`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert subsets_of_size([1, 2, 3], 2) == [[1, 2], [1, 3], [2, 3]]\n    assert subsets_of_size([1, 2, 3], 0) == [[]]\n    assert subsets_of_size([1, 2], 3) == []\n    assert len(subsets_of_size(list(range(10)), 4)) == 210\n    assert \"itertools\" not in source")

### Exercise: Anagrams

Write `anagrams(word)` returning a **sorted** list of every different rearrangement of the letters of `word`. Words with repeated letters have fewer: `"aab"` gives `["aab", "aba", "baa"]`. You may use `itertools.permutations`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert anagrams(\"ab\") == [\"ab\", \"ba\"]\n    assert anagrams(\"aab\") == [\"aab\", \"aba\", \"baa\"]\n    assert anagrams(\"x\") == [\"x\"]\n    assert len(anagrams(\"listen\")) == 720\n    assert len(anagrams(\"banana\")) == 60")

### Exercise: Permutations by hand

Write `perms(s)` that returns every ordering of the characters of the string `s`, as strings, **without** `itertools`. Choose each character in turn to go first, followed by every ordering of the others. For `"abc"` the order is `["abc", "acb", "bac", "bca", "cab", "cba"]`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert perms(\"abc\") == [\"abc\", \"acb\", \"bac\", \"bca\", \"cab\", \"cba\"]\n    assert perms(\"\") == [\"\"]\n    assert len(perms(\"abcdef\")) == 720\n    assert \"itertools\" not in source")

### Exercise: Hitting a target

A shop sells items at the given prices. Write `baskets(prices, budget)` that returns every way to choose **different** items (each at most once) whose prices add up to exactly `budget`, each basket as a list of prices in the original order, with baskets listed in the order the include-first recursion finds them. Prices are all positive, so stop exploring as soon as the total goes over.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert baskets([2, 3, 5, 7], 10) == [[2, 3, 5], [3, 7]]\n    assert baskets([4, 6], 5) == []\n    assert baskets([1, 1, 2], 2) == [[1, 1], [2]]\n    assert baskets([], 0) == [[]]\n    assert len(baskets(list(range(1, 21)), 30)) == 263")

### Exercise: Every meal

A café offers the starters, mains and puddings given as three lists. Write `meals(starters, mains, puddings)` that returns every possible meal as a string like `"soup, curry, cake"`, in the order nested loops would give (all meals with the first starter first). Use `itertools.product`.

In [ ]:
from itertools import product

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    m = meals([\"soup\", \"salad\"], [\"curry\"], [\"cake\", \"fruit\"])\n    assert m == [\"soup, curry, cake\", \"soup, curry, fruit\", \"salad, curry, cake\", \"salad, curry, fruit\"]\n    assert meals([], [\"x\"], [\"y\"]) == []\n    assert \"product\" in source")

### Bug hunt: All the same

This should print all four subsets of `[1, 2]`, but prints `[[], [], [], []]`. Fix it while keeping the shared `chosen` list.

In [ ]:
def subsets(items):
    found = []
    chosen = []
    def go(i):
        if i == len(items):
            found.append(chosen)
            return
        go(i + 1)
        chosen.append(items[i])
        go(i + 1)
        chosen.pop()
    go(0)
    return found

print(subsets([1, 2]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"four different subsets\"\n    expect_output(\"[[], [2], [1], [1, 2]]\")")

### Bug hunt: Nothing at all

This returns `[]` for every input instead of the subsets. Find the one wrong line.

In [ ]:
def subsets(items):
    if not items:
        return []
    rest = subsets(items[1:])
    return rest + [[items[0]] + s for s in rest]

print(subsets(["a", "b"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"four\"\n    expect_output(\"[[], ['b'], ['a'], ['a', 'b']]\")")

### Puzzle: Every ordering

Arrange the lines for `perms(items)`, which returns every ordering of a list. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Count them

Work these out without running them, and write the numbers.

In [ ]:
a = ___   # how many subsets does a list of 6 items have?
b = ___   # how many orderings of 5 different books on a shelf?
c = ___   # how many teams of 2 can be chosen from 5 people?
d = ___   # how many outcomes from tossing a coin 4 times (len(list(product("HT", repeat=4))))?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 64 and b == 120 and c == 10 and d == 16")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a word puzzle

In a "make a word" puzzle you're given some letter tiles and must find every dictionary word you can spell using **some or all** of them (each tile at most once). Write `words_from(tiles, dictionary)` returning a sorted list of the dictionary words that can be made.

Do it with backtracking: build words one letter at a time from the tiles left, and use a set of every **prefix** of the dictionary words to abandon any partial word that no dictionary word starts with. (A brute-force check of every arrangement of every size would work for small inputs, but with 9 tiles there are almost a million arrangements.)

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_small():\n    \"small\"\n    d = [\"at\", \"tea\", \"eat\", \"ate\", \"tee\", \"cat\", \"a\"]\n    assert words_from(\"tea\", d) == [\"a\", \"at\", \"ate\", \"eat\", \"tea\"]\n    assert words_from(\"\", d) == []\n    assert words_from(\"eet\", d) == [\"tee\"]\ndef test_big():\n    \"nine tiles\"\n    d = [\"paint\", \"pint\", \"tin\", \"nap\", \"pat\", \"inapt\", \"patina\", \"tapir\", \"pain\", \"ant\", \"tan\", \"apt\", \"pan\", \"nip\", \"pit\", \"tip\", \"aaa\", \"zip\"]\n    assert words_from(\"patinaxyz\", d) == [\"ant\", \"apt\", \"inapt\", \"nap\", \"nip\", \"pain\", \"paint\", \"pan\", \"pat\", \"patina\", \"pint\", \"pit\", \"tan\", \"tin\", \"tip\", \"zip\"]")

## Warm-up

Each topping is either on the pizza or not: 2 choices, three times, so 2 × 2 × 2 = 8 pizzas (one of them plain). With ten toppings you'd need ten nested loops, and with a list whose length you don't know in advance, nested loops can't do it at all. Recursion can.

## Every subset

A **subset** of a list is any selection of its items, keeping their order: `["a", "c"]` is a subset of `["a", "b", "c"]`, and so are `[]` and the whole list. To list them all, look at the **first** item. Every subset either leaves it out or includes it, and in both cases the rest is any subset of the remaining items:

The base case is the one that surprises people: the empty list has exactly **one** subset, the empty one. Return `[[]]`, not `[]`. Returning `[]` would mean "no subsets at all", and every larger answer built from it would be empty too.

There's a second way to write it that matters more for the rest of today. Walk along the list keeping the choices made so far in `chosen`; at each item, try both ways:

A list of n items has **2ⁿ** subsets: each item doubles the count. Ten items give 1,024; thirty give over a billion.

## Every ordering

A **permutation** is an ordering of all the items. Three people can stand in a queue in 6 ways. To list them: choose who goes **first** (any of them), then put the others in every possible order behind:

n items have **n!** ("n factorial", n × (n−1) × ... × 1) permutations: n choices for first place, n−1 for second, and so on. That grows even faster than 2ⁿ:

Twenty items have about 2.4 × 10¹⁸ orderings. Listing them all at a billion a second would take 77 years. So "try every ordering" only works for small n, a lesson that comes back in Week 17 when we measure how fast programs are.

The same choose-and-continue pattern can build permutations one place at a time with a `chosen` list and a set of what's used:

## Choosing k

A **combination** is a choice of k items where order doesn't matter: picking a team of 2 from Ada, Ben and Cy gives 3 teams, not 6. It's a subset of a fixed size, so use the include-or-skip recursion and stop when you have enough:

The number of ways to choose k from n is written C(n, k), or "n choose k", and Python has it as `math.comb(n, k)`: a lottery choosing 6 balls from 49 has `comb(49, 6)` = 13,983,816 possible tickets.

| | Order matters? | All items? | How many for n items |
|---|---|---|---|
| Subsets | no | any number | 2ⁿ |
| Combinations | no | exactly k | comb(n, k) |
| Permutations | yes | all of them | n! |

## Backtracking

Often you don't want **every** possibility, only those that pass a test: subsets that add up to a target, orderings that fit some rule. Generating everything and filtering works, but it wastes time on choices that were doomed early. **Backtracking** checks as it goes, and abandons a path ("backs up") the moment it can't succeed.

Find every set of coins from a purse that adds up to exactly a target:

The line `if total > target` is the backtracking: once the coins chosen are already too much, adding more can only make it worse (the coins are all positive), so the whole subtree below is skipped. On big inputs this can turn hours into seconds.

The classic backtracking puzzle is placing **queens** on a chessboard so that no two attack each other (no two share a row, column or diagonal). Place one queen per row, and try each column, skipping any column that's already attacked:

`cols[r]` is the column of the queen in row r. Two queens are on the same diagonal when their row distance equals their column distance. The famous 8-queens puzzle has 92 solutions; backtracking finds them all in a fraction of a second, while trying every placement of 8 queens on 64 squares (4 billion) would not.

## itertools

For the standard cases, the `itertools` module has fast, tested versions. They give **tuples**, and produce them one at a time (wrap them in `list()` to see them all):

- `permutations(items, r)`: ordered arrangements of r items (all of them if r is left out).
- `combinations(items, r)`: unordered choices of exactly r.
- `product(a, b, ...)` or `product(a, repeat=k)`: one item from each, like nested loops.

For subsets there's no single function, but `combinations` for every size gives them all:

Use `itertools` when it fits; write your own recursion when you need backtracking or a rule the standard functions don't have.

## Practice

## Quiz

## Daily challenge

## Before you go

List every subset of your own three favourite foods by hand, then check with your function. Count the orderings of the people in your household, and work out how many 4-digit PINs exist (`product(range(10), repeat=4)`). Tomorrow: two classic recursions that move things around, the Towers of Hanoi and flood fill.